[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/lab8/lab8.ipynb)

# Práctica 8: Modelos generativos

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

In [ ]:
import torch
from torchvision import datasets

# Hacemos los imports que sean necesarios
import numpy as np

# Modelos generativos sobre MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [ ]:
import torch
from torchvision import datasets

labeled_data = 0.01 # Tomaremos solo el 1% de las etiquetas
torch.manual_seed(42)

train_dataset = datasets.MNIST(root="./data", train=True, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, download=True)

x_train, y_train = train_dataset.data, train_dataset.targets
x_test, y_test = test_dataset.data, test_dataset.targets

indexes = torch.randperm(len(x_train))
ntrain_data = int(labeled_data * len(x_train))

unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [ ]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)
None

## Modelo generativo

Vamos a crear nuestro propio modelo generativo. En clase de teoría has visto muchas versiones distintas:

1. Mezcla de distribuciones de Gaussianas (GMM)
1. Mezcla de distribuciones multinomiales (Naive Bayes)
1. Modelos de Markov ocultos (HMM)

Tal y como se os apunta en teoría, los modelos generativos abordan un problema más general que la clasificación o regresión: aprenden cómo se estructuran y distribuyen los datos de entrada.

En nuestro caso, vamos a modelar los datos de entrada mediante el uso de **Autoencoders**. 

# Autoencoders

El autoencoder es un tipo de red que se utiliza para aprender codificaciones eficientes de datos sin etiquetar (lo que se conoce como aprendizaje no supervisado). Es una red que tiene el mismo tamaño en la entrada como en la salida, puesto que el objetivo de la red es reconstruir la entrada con la menor pérdida posible.

Si lo que hacemos es reconstruir la entrada, ¿qué sentido tiene el usar la red? Habitualmente, **la red consta, a su mitad, de una capa con menos elementos que los datos de entrada**. Por tanto, al reconstruir los datos de la entrada a la salida, en esa capa tendremos una versión *comprimida* de la entrada, que contendrá la mayor parte de su información.

Por tanto, podemos dividir un autoencoder en 3 secciones diferentes, tal y como se ve en la siguiente figura:

![](https://drive.google.com/uc?export=view&id=1yxkKZV0J0YplQAGPGJxQ2Z80Ad6L94eu)

1. **Encoder:** es la parte inicial de la red, encargada de comprimir los datos de la entrada.
1. **Code:** es la salida del encoder, contiene la versión *comprimida* de los datos de entrada.
1. **Decoder:** se encarga de, partiendo de la salida del *Encoder*, reconstruir la red.

## Crea tu propio Autoencoder

El diseño del autoencoder es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [ ]:
# TODO: crea tu propio autoencoder

class MiAutoencoder:

    def __init__(self, input_shape):
        # TODO : define el modelo
        None
    
    def fit(self, X, y=None, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        None

    def get_encoded_data(self, X):
        # TODO: devuelve la salida del encoder (code)
        None
        
    def __del__(self):
        if hasattr(self, "model"):
            del self.model
        torch.cuda.empty_cache()  # libera memoria GPU

## Crea tu propio Clasificador

A continuación crearemos un clasificador que sea capaz de predecir la etiqueta pero no a partir de los datos originales, sino de la codificación `code` aprendida por el autoencoder. El diseño del clasificador es libre, pero recuerda que tiene que ser simple (máximo dos capas). **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [ ]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self, optimizer):
        # TODO : define el modelo
        self.model = None

        # TODO: define la función de pérdida
        self.criterion = None

        # TODO: crea el optimizador (usa el parámetro optimizer si quieres)
        self.optimizer = None

        # TODO: define el dispositivo (cpu o cuda)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if self.model is not None:
            self.model.to(self.device)
    
    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo (loop de epochs + batches)
        pass

    def predict(self, X):
        # TODO: devuelve la clase ganadora (argmax)
        pass
    
    def predict_proba(self, X):
        # TODO: devuelve las probabilidades (softmax)
        pass
    
    def score(self, X, y):
        # TODO: calcula el accuracy
        pass

    def __del__(self):
        if hasattr(self, "model"):
            del self.model
        torch.cuda.empty_cache()  # libera memoria GPU

### Entrenamiendo del modelo sin supervisar

Primero de todo, a modo de comparación, crea un modelo (de capacidad similar a tu encoder+clasificador) que puedas entrenar supervisadamente con `x_train` e `y_train`. Anota su rendimiento.

In [ ]:
# TODO - Crea un modelo, entrénalo con x_train e y_train y muestra su rendimiento en test.

### Entrenamiendo del modelo semisupervisado

El entrenamiento del sistema semisupervisado se realiza en dos pasos.

1. Se entrena el autoencoder con todos los datos (etiquetados y sin etiquetar).
1. Se entrena un clasificador simple (una o dos capas), teniendo como entrada la salida del encoder (**code**) de los datos etiquetados.

<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar las funciones que hemos definido en el autoencoder y en el clasificador.

In [ ]:
# TODO: implementa el algoritmo semisupervised_training.

def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):
    None

### Entrenamos nuestro modelo

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada.

In [ ]:
# Crea tu autoencoder y tu clasificador
autoencoder = None
classifier = None

In [ ]:
# TODO: Entrena tu modelo
None

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
pred_data = autoencoder.get_encoded_data(x_test)
print('Test accuracy :', classifier.score(pred_data, y_test))

## Mejorando el código

nuestro modelo actual requiere de dos pasos para entrenarse, pero podría realizarse en un único paso si **creamos un modelo con las dos salidas (autoencoder y clasificador)**. 

Para ello, hay que tener en cuenta que, en los datos sin etiquetar, su contribución al clasificador debería ser nula.


### TRABAJO: Crea el nuevo modelo y modifica la función semisupervised_training para tener en cuenta todos los puntos mencionados anteriormente

In [ ]:
# TODO: crea el nuevo modelo

# TODO: crea tu propio clasificador

class MiClasificadorSemisupervisado:

    def __init__(self, optimizer):
        # TODO : define el modelo
        self.model = None

        # TODO: define la función de pérdida
        self.criterion = None

        # TODO: crea el optimizador (usa el parámetro optimizer si quieres)
        self.optimizer = None

        # TODO: define el dispositivo (cpu o cuda)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if self.model is not None:
            self.model.to(self.device)
    
    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo (loop de epochs + batches)
        pass

    def predict(self, X):
        # TODO: devuelve la clase ganadora (argmax)
        pass
    
    def predict_proba(self, X):
        # TODO: devuelve las probabilidades (softmax)
        pass
    
    def score(self, X, y):
        # TODO: calcula el accuracy
        pass

    def __del__(self):
        if hasattr(self, "model"):
            del self.model
        torch.cuda.empty_cache()  # libera memoria GPU

In [ ]:
# TODO: reescribe la función semisupervised_training para incorporar las mejoras mencionadas anteriormente

def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    None

In [ ]:
# TODO: Crea y entrena tu clasificador
model = None

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
print('Test accuracy :', model.score(x_test, y_test))

# Hay vida más allá del autoencoder

Existen otros métodos para conseguir representaciones útiles de los datos sin apoyarse en la reconstrucción. A continuación implementarás un método para aprender representaciones que verifiquen que:
    - Dos variantes de la misma imagen tienen la misma representación
    - Dos imágenes distintas tienen distinta representación
Para ello, para cada imagen generaremos dos variantes y, usando una red convolucional, calcularemos sus representaciones. La función de pérdida penalizará representaciones iguales para elementos distintos del lote, mientras que también penalizará diferencias entre las representaciones de las dos variantes de una misma imagen. Llamamos a este método [*aprendizaje contrastivo*](https://arxiv.org/abs/2002.05709). En este ejemplo usaremos además una salida que debe llevar las representaciones a clusters bien diferenciados (esto favorece representaciones con información relevante). Impleméntalo siguiendo estas pautas:

1. Define un modelo $model$ convolucional similar al encoder de un autoencoder (la entrada es el tamaño de la imagen, la salida el vector de representación)
1. Define una capa de salida $cluster$ que, partiendo de la salida de model, nos devuelva una salida con el mismo número de clases que el dataset a utilizar (la entrada es el vector de representación), usando softmax como activación de salida
1. Para cada batch de entrenamiento $X$:  *# Usa un tamaño de batch alto, mínimo 128*
    1. Modifica las imágenes de entrada con [data_augmentation](https://docs.pytorch.org/vision/0.21/transforms.html), llámala $augX_1$.
    1. Modifica otra vez las imágenes de entrada con otra transformación distinta (*data_augmentation_2*), llámala $augX_2$.
    1. $augX_{1comp} \leftarrow model(augX_1)$ *# Usa [normalize](https://docs.pytorch.org/docs/2.11/generated/torch.nn.functional.normalize.html) a la salida de tu modelo para que las representaciones tengan siempre la misma escala*
    1. $augX_{2comp} \leftarrow model(augX_2)$
    1. $M \leftarrow augX_{1comp} ~ augX_{2comp}^T$ *# Esto calcula el producto escalar (similitud coseno) de la representación de la variante 1 de cada elemento del lote con la representación de la variante 2 de cada uno de los elementos del lote. M será una matriz de similitudes donde cada elemento (i,j) representa la similitud de la representación de la variante 1 del elemento i con la de la variante 2 del elemento j*
    1. $loss_M \leftarrow torch.nn.functional.cross\_entropy(M/ \tau, torch.arange(batch\_size))$ *#Los pares (i,j) deben ser 1 si i==j y 0 en caso contrario*
        1. $\tau$ es un hiperparámetro que se suele definir a 5.0
    1. $cX_{1comp} \leftarrow cluster(augX_{1comp})$ *# Otra red debe llevar cada representación a un cluster*
    1. $cX_{2comp} \leftarrow cluster(augX_{2comp})$
    1. $loss_C \leftarrow cX_{1comp}(1 - cX_{1comp}) + cX_{2comp}(1 - cX_{2comp})$ *# Las salidas de la red cluster deben ser extremas (cerca de 0 o 1, no a mitad del rango)*
    1. $loss \leftarrow loss_M + \lambda~loss_C$
        1. $\lambda$ es un hiperparámetro (puedes probar con 0.5)


In [ ]:
# Escribe aquí la solución. Crea tantos bloques de código como necesites. Puedes utilizar la siguiente red para generar distorsiones

import torchvision.transforms as transforms

data_augmentation = transforms.Compose([
    # transforms.RandomHorizontalFlip(),  # Puede ser util en otros casos
    transforms.RandomRotation(0.05 * 180),  # en grados
    transforms.RandomAffine(
        degrees=0,
        translate=(0.15, 0.15),
        scale=(0.85, 1.15)  # approx para zoom ±15%
    ),
])

data_augmentation_2 = transforms.Compose([
    # transforms.RandomHorizontalFlip(),  # Puede ser util en otros casos
    transforms.RandomAffine(
        degrees=0,
        translate=(0.15, 0.15)
    ),
    transforms.Resize((48, 48)),   # MNIST usar (40, 40)
    transforms.RandomCrop((32, 32))  # MNIST usar (28, 28)
])


# ¡ENHORABUENA! Has completado la práctica de modelos generativos.


# Trabajo extra

¿Has probado a hacer el autoencoder totalmente convolucional? Para el *decoder* puedes usar las funciones [UpSample](https://docs.pytorch.org/docs/2.11/generated/torch.nn.Upsample.html) o [ConvTranspose2d](https://docs.pytorch.org/docs/2.11/generated/torch.nn.ConvTranspose2d.html).